
> **Reference:** VanderPlas, Jake. **Python data science handbook**: Essential tools for working with data. " O'Reilly Media, Inc.", 2016.


# CHAPTER 2 - PART 1


**Efficient storage and manipulation of numerical arrays is absolutely fundamental to the process of doing data science**. We'll now take a look at the specialized tools that Python has for handling such numerical arrays: the **NumPy package** and the **Pandas package** (discussed in Chapter 3.)

This chapter will cover NumPy in detail. **NumPy (short for Numerical Python)** provides an efficient interface to store and operate on dense data buffers. In some ways, NumPy arrays are like Python's built-in `list` type, but **NumPy arrays provide much more efficient storage and data operations** as the arrays grow larger in size. NumPy arrays form the core of nearly the entire ecosystem of data science tools in Python, so time spent learning to use NumPy effectively will be valuable no matter what aspect of data science interests you.

In [1]:
import numpy as np
np.__version__

'2.0.2'

---
# 1- Understanding Data Types in Python

Effective data-driven science and computation requires **understanding how data is stored and manipulated**. This section outlines and contrasts how arrays of data are handled in the Python language itself, and how NumPy improves on this. Understanding this difference is fundamental to understanding much of the material throughout the rest of the book.

Users of Python are often drawn in by its **ease of use**, one piece of which is **dynamic typing**. While a statically typed language like C or Java requires each variable to be explicitly declared, a dynamically typed language like Python skips this specification. For example, in C you might specify a particular operation as follows:

```c
/* C code */
int result = 0;
for(int i=0; i<100; i++){
    result += i;
}
```

While in Python the equivalent operation could be written this way:

In [2]:
# Python code
result = 0
for i in range(100):
    result += i

Notice the main difference: in C, the **data types of each variable are explicitly declared**, while in Python the **types are dynamically inferred**. This means, for example, that we can assign any kind of data to any variable:

In [3]:
# Python code
x = 4
x = "four"

Here we've switched the contents of `x` from an integer to a string. The same thing in C would lead (depending on compiler settings) to a compilation error or other unintended consequences:

```c
/* C code */
int x = 4;
x = "four"; // FAILS
```

This sort of **flexibility** is one piece that makes Python and other dynamically typed languages **convenient and easy to use**. Understanding how this works is an important piece of learning to analyze data efficiently and effectively with Python. But what this type flexibility also points to is the fact that **Python variables are more than just their value**; they also contain **extra information about the type of the value**. We'll explore this more in the sections that follow.

### A Python Integer Is More Than Just an Integer

The standard Python implementation is **written in C**. This means that every Python object is simply a cleverly disguised **C structure**, which contains not only its value, but other information as well. For example, when we define an integer in Python, such as `x = 10000`, `x` is not just a "raw" integer. It's actually a **pointer to a compound C structure**, which contains several values. Looking through the Python 3.4 source code, we find that the integer (long) type definition effectively looks like this (once the C macros are expanded):

```c
struct _longobject {
    long ob_refcnt;
    PyTypeObject *ob_type;
    size_t ob_size;
    long ob_digit[1];
};
```

This means that there is some **overhead in storing an integer in Python** as compared to an integer in a compiled language like C, as illustrated in the following Figure.



### A Python List Is More Than Just a List

Let's consider now what happens when we use a Python data structure that holds many Python objects. The standard **mutable multielement container** in Python is the `list`. We can create a list of integers as follows:

In [4]:
L = list(range(10))
L

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [5]:
print(type(L))
print(type(L[0]))

<class 'list'>
<class 'int'>


Or, similarly, a list of strings:

In [6]:
L2 = [str(c) for c in L]
L2

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

In [7]:
type(L2[0])

str

Because of Python's **dynamic typing**, we can even create **heterogeneous lists**:

In [8]:
L3 = [True, "2", 3.0, 4]
[type(item) for item in L3]

[bool, str, float, int]

But this **flexibility comes at a cost**: to allow these flexible types, each item in the list must contain its own type info, reference count, and other information—that is, **each item is a complete Python object**. In the special case that all variables are of the same type, much of this information is redundant: it can be **much more efficient to store data in a fixed-type array**.


---
# 2- Creating NumPy Arrays

## Creating Arrays from Python Lists

First, we can use `np.array` to create arrays from Python lists:

In [9]:
# Integer array:
np.array([1, 4, 2, 5, 3])

array([1, 4, 2, 5, 3])

Remember that **unlike Python lists, NumPy is constrained to arrays that all contain the same type**. If types do not match, NumPy will **upcast if possible** (here, integers are upcast to floating point):

In [10]:
np.array([3.14, 4, 2, 3])

array([3.14, 4.  , 2.  , 3.  ])

If we want to **explicitly set the data type** of the resulting array, we can use the `dtype` keyword:

In [11]:
np.array([1, 2, 3, 4], dtype='float32')

array([1., 2., 3., 4.], dtype=float32)

Finally, unlike Python lists, **NumPy arrays can explicitly be multidimensional**; here's one way of initializing a multidimensional array using a list of lists:

In [12]:
# Nested lists result in multidimensional arrays
np.array([range(i, i + 3) for i in [2, 4, 6]])

array([[2, 3, 4],
       [4, 5, 6],
       [6, 7, 8]])

The inner lists are treated as **rows of the resulting two-dimensional array**.

---
## Creating Arrays from Scratch

Especially for larger arrays, it is more efficient to **create arrays from scratch** using routines built into NumPy. Here are several examples:

In [13]:
# Create a length-10 integer array filled with zeros
np.zeros(10, dtype=int)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [14]:
# Create a 3x5 floating-point array filled with 1s
np.ones((3, 5), dtype=float)

array([[1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.]])

In [15]:
# Create a 3x5 array filled with 3.14
np.full((3, 5), 3.14)

array([[3.14, 3.14, 3.14, 3.14, 3.14],
       [3.14, 3.14, 3.14, 3.14, 3.14],
       [3.14, 3.14, 3.14, 3.14, 3.14]])

In [16]:
# Create an array filled with a linear sequence
# Starting at 0, ending at 20 excluded, stepping by 2
# (this is similar to the built-in range() function)
np.arange(0, 20, 2)

array([ 0,  2,  4,  6,  8, 10, 12, 14, 16, 18])

In [17]:
# Create an array of five values evenly spaced between 0 and 1
np.linspace(0, 1, 5)

array([0.  , 0.25, 0.5 , 0.75, 1.  ])

In [18]:
# Create a 3x3 array of 'uniformly' distributed
# random values between 0 and 1
np.random.random((3, 3))

array([[3.42967950e-01, 3.10027886e-04, 4.05892159e-01],
       [8.13247707e-01, 6.76260165e-01, 5.97870787e-03],
       [7.51640979e-02, 2.65737320e-01, 1.61965569e-01]])

In [19]:
# Create a 3x3 array of 'normally' distributed random values
# with mean 0 and standard deviation 1
np.random.normal(0, 1, (3, 3))

array([[-0.41158229,  0.21995984,  1.08573866],
       [ 0.08168678, -0.66958075,  0.15569462],
       [ 0.34188159,  0.85435564, -0.00267304]])

In [20]:
# Create a 3x3 array of 'random integers' in the interval [0, 10)
np.random.randint(0, 10, (3, 3))

array([[0, 6, 3],
       [4, 2, 5],
       [8, 0, 3]])

In [21]:
# Create a 3x3 identity matrix
np.eye(3)

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

In [22]:
# Create an uninitialized array of three integers
# The values will be whatever happens to already exist at that
# memory location
np.empty(3)

array([1., 1., 1.])

---
## NumPy Standard Data Types

NumPy arrays contain values of a **single type**, so it is important to have detailed knowledge of those types and their limitations. Because NumPy is built in C, the types will be familiar to users of C, Fortran, and other related languages.

The standard NumPy data types are listed in **Table 2-1**. Note that when constructing an array, you can specify them using a string:
```python
np.zeros(10, dtype='int16')
```

Or using the associated NumPy object:
```python
np.zeros(10, dtype=np.int16)
```

### Table 2-1. Standard NumPy data types

| Data type | Description |
|-----------|-------------|
| `bool_` | Boolean (True or False) stored as a byte |
| `int_` | Default integer type (same as C long; normally either int64 or int32) |
| `intc` | Identical to C int (normally int32 or int64) |
| `intp` | Integer used for indexing (same as C ssize_t; normally either int32 or int64) |
| `int8` | Byte (–128 to 127) |
| `int16` | Integer (–32768 to 32767) |
| `int32` | Integer (–2147483648 to 2147483647) |
| `int64` | Integer (–9223372036854775808 to 9223372036854775807) |
| `uint8` | Unsigned integer (0 to 255) |
| `uint16` | Unsigned integer (0 to 65535) |
| `uint32` | Unsigned integer (0 to 4294967295) |
| `uint64` | Unsigned integer (0 to 18446744073709551615) |
| `float_` | Shorthand for float64 |
| `float16` | Half-precision float: sign bit, 5 bits exponent, 10 bits mantissa |
| `float32` | Single-precision float: sign bit, 8 bits exponent, 23 bits mantissa |
| `float64` | Double-precision float: sign bit, 11 bits exponent, 52 bits mantissa |
| `complex_` | Shorthand for complex128 |
| `complex64` | Complex number, represented by two 32-bit floats |
| `complex128` | Complex number, represented by two 64-bit floats |

More advanced type specification is possible, such as specifying big or little endian numbers; for more information, refer to the NumPy documentation.

---
# 3- Basics of NumPy Arrays

Data manipulation in Python is nearly synonymous with **NumPy array manipulation**: even newer tools like Pandas (Chapter 3) are built around the NumPy array. This section will present several examples using NumPy array manipulation to **access data and subarrays**, and to **split, reshape, and join** the arrays. While the types of operations shown here may seem a bit dry and pedantic, they comprise the **building blocks** of many other examples used throughout the book. Get to know them well!

We'll cover a few categories of basic array manipulations here:

**Attributes of arrays**
- Determining the size, shape, memory consumption, and data types of arrays

**Indexing of arrays**
- Getting and setting the value of **individual** array elements

**Slicing of arrays**
- Getting and setting **smaller subarrays** within a larger array

**Reshaping of arrays**
- Changing the shape of a given array

**Joining and splitting of arrays**
- Combining multiple arrays into one, and splitting one array into many

## (1) Array Attributes

First let's discuss some useful **array attributes**. We'll start by defining three random arrays: a one-dimensional, two-dimensional, and three-dimensional array. We'll use NumPy's **random number generator**, which we will seed with a set value in order to ensure that the same random arrays are generated each time this code is run:

In [23]:
import numpy as np
np.random.seed(0)  # Seed for reproducibility

x1 = np.random.randint(10, size=6)  # One-dimensional array
x2 = np.random.randint(10, size=(3, 4))  # Two-dimensional array
x3 = np.random.randint(10, size=(3, 4, 5))  # Three-dimensional array

Each array has attributes `ndim` (the **number of dimensions**), `shape` (the **size of each dimension**), and `size` (the **total size of the array**):

In [24]:
print("x3 ndim: ", x3.ndim)
print("x3 shape:", x3.shape)
print("x3 size: ", x3.size)

x3 ndim:  3
x3 shape: (3, 4, 5)
x3 size:  60


Another useful attribute is the `dtype`, the **data type of the array**:

In [25]:
print("dtype:", x3.dtype)

dtype: int64


Other attributes include `itemsize`, which lists the **size (in bytes) of each array element**, and `nbytes`, which lists the **total size (in bytes) of the array**:

In [26]:
print("itemsize:", x3.itemsize, "bytes")
print("nbytes:", x3.nbytes, "bytes")

itemsize: 8 bytes
nbytes: 480 bytes


In general, we expect that `nbytes` is equal to `itemsize` times `size`.

---
## (2) Array Indexing: Accessing Single Elements

If you are familiar with Python's standard list indexing, indexing in NumPy will feel quite familiar. In a **one-dimensional array**, you can access the i<sup>th</sup> value (counting from zero) by specifying the desired index in square brackets, just as with Python lists:

In [27]:
x1

array([5, 0, 3, 3, 7, 9])

In [28]:
x1[0]

np.int64(5)

In [29]:
x1[4]

np.int64(7)

To **index from the end of the array**, you can use **negative indices**:

In [30]:
x1[-1]  # The last element

np.int64(9)

In [31]:
x1[-2]  # The one before the last

np.int64(7)

In a **multidimensional array**, you access items using a **comma-separated tuple of indices**:

In [32]:
x2

array([[3, 5, 2, 4],
       [7, 6, 8, 8],
       [1, 6, 7, 7]])

In [33]:
x2[0, 0]

np.int64(3)

In [34]:
x2[2, 0]

np.int64(1)

In [35]:
x2[2, -1]

np.int64(7)

You can also **modify values** using any of the above index notation:

In [36]:
x2[0, 0] = 12
x2

array([[12,  5,  2,  4],
       [ 7,  6,  8,  8],
       [ 1,  6,  7,  7]])

Keep in mind that, **unlike Python lists, NumPy arrays have a fixed type**. This means, for example, that if you attempt to insert a floating-point value to an integer array, the value will be **silently truncated**. Don't be caught unaware by this behavior!

In [37]:
x1

array([5, 0, 3, 3, 7, 9])

In [38]:
x1[0] = 3.14159  # This will be truncated!
x1

array([3, 0, 3, 3, 7, 9])

---
## (3) Array Slicing: Accessing Subarrays

Just as we can use square brackets to access individual array elements, we can also use them to access **subarrays** with the **slice notation**, marked by the colon (`:`) character. The NumPy slicing syntax follows that of the standard Python list; to access a slice of an array `x`, use this:

```python
x[start:stop:step]
```

If any of these are unspecified, they default to the values `start=0`, `stop=size of dimension`, `step=1`. We'll take a look at accessing subarrays in one dimension and in multiple dimensions.

### One-dimensional subarrays

In [39]:
x = np.arange(10)
x

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [40]:
x[:5]  # First five elements (the stop, index 5, is excluded)

array([0, 1, 2, 3, 4])

In [41]:
x[5:]  # Elements at index 5 and after

array([5, 6, 7, 8, 9])

In [42]:
x[4:7]  # Middle subarray (the stop, index 7, is exluded)

array([4, 5, 6])

In [43]:
x[::2]  # Every other element --> step = 2
# [start:stop:step]
# --> here start and stop is empty to the whole range,
# with step = 2

array([0, 2, 4, 6, 8])

In [44]:
x[1::2]  # Every other element, starting at index 1
# [start:stop:step]
# --> here start = 1 and stop is empty, so starting at ind 1 to the end
# with step = 2

array([1, 3, 5, 7, 9])

A potentially confusing case is when the **step value is negative**. In this case, the defaults for `start` and `stop` are swapped. This becomes a convenient way to **reverse an array**:

In [45]:
x[::-1]  # all elements, reversed

array([9, 8, 7, 6, 5, 4, 3, 2, 1, 0])

In [46]:
x[5::-2]  # reversed every other from index 5

array([5, 3, 1])

When `step < 0`, the defaults are:

- `start` **defaults to: The end of the axis**. Specifically, the index of the last element (index -1 or length - 1).

- `stop` **defaults to: The "beginning" of the axis**. Specifically, it extends past index 0 to include the very first element. In internal logic, this is often represented as being effectively "before the first element."

### Multidimensional subarrays

Multidimensional slices work in the same way, with **multiple slices separated by commas**. For example:

In [47]:
x2

array([[12,  5,  2,  4],
       [ 7,  6,  8,  8],
       [ 1,  6,  7,  7]])

In [48]:
x2[:2, :3]  # The first two rows, the first three columns

array([[12,  5,  2],
       [ 7,  6,  8]])

In [49]:
x2[:3, ::2]  # All rows, every other column

array([[12,  2],
       [ 7,  8],
       [ 1,  7]])

Finally, subarray dimensions can even be **reversed together**:

In [50]:
x2[::-1, ::-1]

array([[ 7,  7,  6,  1],
       [ 8,  8,  6,  7],
       [ 4,  2,  5, 12]])

### Accessing array rows and columns

One commonly needed routine is **accessing single rows or columns** of an array. You can do this by combining indexing and slicing, using an **empty slice** marked by a single colon (`:`):

In [51]:
x2

array([[12,  5,  2,  4],
       [ 7,  6,  8,  8],
       [ 1,  6,  7,  7]])

In [52]:
x2[:, 0]  # Column 0 --> returns 1D array

array([12,  7,  1])

In [53]:
x2[0, :]  # Row 0 --> returns 1D array

array([12,  5,  2,  4])

**In the case of `row access`, the empty slice can be omitted for a more compact syntax**:

In [54]:
x2[0]  # Equivalent to x2[0, :]

array([12,  5,  2,  4])

### Subarrays as no-copy views

One important—and extremely useful—thing to know about array slices is that they return **views rather than copies** of the array data. This is one area in which NumPy array slicing differs from Python list slicing: in lists, slices will be copies. Consider our two-dimensional array from before:

In [55]:
print(x2)

[[12  5  2  4]
 [ 7  6  8  8]
 [ 1  6  7  7]]


Let's extract a 2×2 subarray from this:

In [56]:
x2_sub = x2[:2, :2]  # First two rows, first two cols
print(x2_sub)

[[12  5]
 [ 7  6]]


Now if we **modify this subarray**, we'll see that the **original array is changed**! Observe:

In [57]:
x2_sub[0, 0] = 99
print(x2_sub)

[[99  5]
 [ 7  6]]


In [58]:
print(x2)

[[99  5  2  4]
 [ 7  6  8  8]
 [ 1  6  7  7]]


This default behavior is actually quite useful: it means that when we work with **large datasets**, we can access and process pieces of these datasets **without the need to copy the underlying data buffer**.

### Creating copies of arrays

Despite the nice features of array views, it is sometimes useful to instead **explicitly copy** the data within an array or a subarray. This can be most easily done with the `copy()` method:

In [59]:
x2_sub_copy = x2[:2, :2].copy()
print(x2_sub_copy)

[[99  5]
 [ 7  6]]


If we now modify this subarray, the **original array is not touched**:

In [60]:
x2_sub_copy[0, 0] = 42
print(x2_sub_copy)

[[42  5]
 [ 7  6]]


In [61]:
print(x2)

[[99  5  2  4]
 [ 7  6  8  8]
 [ 1  6  7  7]]


---
## (4) Array Reshaping

Another useful type of operation is **reshaping of arrays**. The most flexible way of doing this is with the `reshape()` method. For example, if you want to put the numbers 1 through 9 in a 3×3 grid, you can do the following:

In [62]:
grid = np.arange(1, 10).reshape((3, 3))
print(grid)

[[1 2 3]
 [4 5 6]
 [7 8 9]]


Note that for this to work, the **size of the initial array must match the size of the reshaped array**. Where possible, the `reshape` method will use a no-copy view of the initial array, but with noncontiguous memory buffers this is not always the case.

Another common reshaping pattern is the **conversion of a one-dimensional array into a two-dimensional row or column matrix**. You can do this with the `reshape` method, or more easily by making use of the `newaxis` keyword within a slice operation:

In [63]:
x = np.array([1, 2, 3])  # Shape: 1D (3,)
x

array([1, 2, 3])

In [64]:
# Row vector via reshape
x.reshape((1, 3))  # Shape: 2D (1, 3)

array([[1, 2, 3]])

In [65]:
# Row vector via newaxis
x[np.newaxis, :]  # Shape: 2D (1, 3)

array([[1, 2, 3]])

In [66]:
# Column vector via reshape
x.reshape((3, 1))  # Shape: 2D (3, 1)

array([[1],
       [2],
       [3]])

In [67]:
# Column vector via newaxis
x[:, np.newaxis]  # Shape: 2D (3, 1)

array([[1],
       [2],
       [3]])

When you use `np.newaxis` in a slicing operation, NumPy inserts a **new axis of length 1** at that position.

---
## (5) Array Concatenation and Splitting

All of the preceding routines worked on single arrays. It's also possible to **combine multiple arrays into one**, and to conversely **split a single array into multiple arrays**. We'll take a look at those operations here.

### Concatenation of arrays

Concatenation, or joining of two arrays in NumPy, is primarily accomplished through the routines `np.concatenate`, `np.vstack`, and `np.hstack`. `np.concatenate` takes a **tuple or list of arrays** as its first argument, as we can see here:

In [68]:
x = np.array([1, 2, 3])  # 1D (3,)
y = np.array([3, 2, 1])  # 1D (3,)
np.concatenate([x, y])   # 1D (6,)

array([1, 2, 3, 3, 2, 1])

You can also **concatenate more than two arrays** at once:

In [69]:
z = [99, 99, 99]
print(np.concatenate([x, y, z]))

[ 1  2  3  3  2  1 99 99 99]


`np.concatenate` can also be used for **two-dimensional arrays**:

In [70]:
grid = np.array([[1, 2, 3],
                 [4, 5, 6]])  # Shape (2, 3)
print(grid.shape)

(2, 3)


In [71]:
# Concatenate along the first axis, axis=0 by default
# meaning more rows
np.concatenate([grid, grid])  # Shape: (*4, 3)

array([[1, 2, 3],
       [4, 5, 6],
       [1, 2, 3],
       [4, 5, 6]])

In [72]:
# Equivalent to:
np.concatenate([grid, grid], axis=0)  # Shape: (*4, 3)

array([[1, 2, 3],
       [4, 5, 6],
       [1, 2, 3],
       [4, 5, 6]])

In [73]:
# Concatenate along the second axis (zero-indexed)
# meaning more cols
np.concatenate([grid, grid], axis=1)  # Shape: (2, *6)

array([[1, 2, 3, 1, 2, 3],
       [4, 5, 6, 4, 5, 6]])

For working with arrays of **mixed dimensions**, it can be clearer to use the `np.vstack` (vertical stack = more rows) and `np.hstack` (horizontal stack = more cols) functions:

In [74]:
x = np.array([1, 2, 3])  # Shape: 1D (3,)
grid = np.array([[9, 8, 7],
                 [6, 5, 4]])  # Shape: (2, 3)

# Vertically stack the arrays
np.vstack([x, grid])  # Shape: (3, 3)

array([[1, 2, 3],
       [9, 8, 7],
       [6, 5, 4]])

In [75]:
x2 = np.array([[1, 2, 3]])  # Shape: (1, 3)
grid = np.array([[9, 8, 7],
                 [6, 5, 4]])  # Shape: (2, 3)

# Vertically stack the arrays
np.vstack([x2, grid])  # Shape: (3, 3) from (1, 3) with (2, 3)

array([[1, 2, 3],
       [9, 8, 7],
       [6, 5, 4]])

In [76]:
# Horizontally stack the arrays
y = np.array([[99],
              [99]])  # Shape: (2, 1)
np.hstack([grid, y])  # Shape: (2, 4) from (2, 3) with (2, 1)

array([[ 9,  8,  7, 99],
       [ 6,  5,  4, 99]])

### Splitting of arrays

The opposite of concatenation is **splitting**, which is implemented by the functions `np.split`, `np.hsplit`, and `np.vsplit`. For each of these, we can pass a **list of indices** giving the **split points**:

In [77]:
x = [1, 2, 3, 99, 99, 3, 2, 1]
x1, x2, x3 = np.split(x, [3, 5])
print(x1, x2, x3)

[1 2 3] [99 99] [3 2 1]


Notice that **N split points lead to N + 1 subarrays**. The related functions `np.hsplit` and `np.vsplit` are similar:

In [78]:
grid = np.arange(16).reshape((4, 4))
grid

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [12, 13, 14, 15]])

In [79]:
# Vertical split = less rows
upper, lower = np.vsplit(grid, [2])
print(upper)
print(lower)

[[0 1 2 3]
 [4 5 6 7]]
[[ 8  9 10 11]
 [12 13 14 15]]


In [80]:
# Horizontal split = less cols
left, right = np.hsplit(grid, [2])
print(left)
print(right)

[[ 0  1]
 [ 4  5]
 [ 8  9]
 [12 13]]
[[ 2  3]
 [ 6  7]
 [10 11]
 [14 15]]


In [81]:
# Vertical split = less rows
# Multiple split points
upper2, middle2, lower2 = np.vsplit(grid, [2, 3])
print(upper2)
print(middle2)
print(lower2)

[[0 1 2 3]
 [4 5 6 7]]
[[ 8  9 10 11]]
[[12 13 14 15]]


---
# 4- Universal Functions (UFuncs)


Computation on NumPy arrays **can be very fast, or it can be very slow**. The key to making it fast is to use **vectorized operations**, generally implemented through NumPy's **universal functions (ufuncs)**. This section motivates the need for NumPy's ufuncs, which can be used to make repeated calculations on array elements **much more efficient**. It then introduces many of the most common and useful arithmetic ufuncs available in the NumPy package.

## The Slowness of Loops

Python's default implementation (known as **CPython**) does some operations very slowly. This is in part due to the **dynamic, interpreted nature** of the language: the fact that types are flexible, so that sequences of operations cannot be compiled down to efficient machine code as in languages like C and Fortran.

Imagine we have an array of values and we'd like to compute the reciprocal of each. A straightforward approach might look like this:

In [82]:
import numpy as np
np.random.seed(0)

def compute_reciprocals(values):
    output = np.empty(len(values))
    for i in range(len(values)):
        output[i] = 1.0 / values[i]
    return output

values = np.random.randint(1, 10, size=5)
print(values)
compute_reciprocals(values)

[6 1 4 4 8]


array([0.16666667, 1.        , 0.25      , 0.25      , 0.125     ])

This implementation probably feels fairly natural to someone from, say, a C or Java background. But if we measure the execution time of this code for a large input, we see that this operation is **very slow**, perhaps surprisingly so! We'll benchmark this with IPython's `%timeit` magic:

In [83]:
big_array = np.random.randint(1, 100, size=1000000)
%timeit compute_reciprocals(big_array)

The slowest run took 4.14 times longer than the fastest. This could mean that an intermediate result is being cached.
4.04 s ± 2.09 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


It turns out that the **bottleneck here is not the operations themselves**, but the **type-checking and function dispatches** that CPython must do at each cycle of the loop. Each time the reciprocal is computed, Python first examines the object's type and does a dynamic lookup of the correct function to use for that type. If we were working in compiled code instead, this type specification would be known before the code executes and the result could be computed much more efficiently.

## Introducing UFuncs

For many types of operations, NumPy provides a convenient interface into just this kind of **statically typed, compiled routine**. This is known as a **vectorized operation**. You can accomplish this by simply **performing an operation on the array, which will then be applied to each element**. This vectorized approach is designed to **push the loop into the compiled layer** that underlies NumPy, **leading to much faster execution**.

Compare the results of the following two:

In [84]:
print(compute_reciprocals(values))
print(1.0 / values)

[0.16666667 1.         0.25       0.25       0.125     ]
[0.16666667 1.         0.25       0.25       0.125     ]


Looking at the execution time for our big array, we see that it completes **orders of magnitude faster** than the Python loop:

In [85]:
%timeit (1.0 / big_array)

2 ms ± 266 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


Vectorized operations in NumPy are implemented via **ufuncs**, whose main purpose is to quickly execute repeated operations on values in NumPy arrays. Ufuncs are **extremely flexible**—before we saw an operation between a scalar and an array, but we can also operate between two arrays:

In [86]:
print(np.arange(5))
print(np.arange(1, 6))
np.arange(5) / np.arange(1, 6)  # Element-wise division

[0 1 2 3 4]
[1 2 3 4 5]


array([0.        , 0.5       , 0.66666667, 0.75      , 0.8       ])

And ufunc operations are **not limited to one-dimensional arrays**—they can act on multidimensional arrays as well:

In [87]:
x = np.arange(9).reshape((3, 3))
print(x)
2 ** x  # For each element, 2 power the element

[[0 1 2]
 [3 4 5]
 [6 7 8]]


array([[  1,   2,   4],
       [  8,  16,  32],
       [ 64, 128, 256]])

Computations using vectorization through ufuncs are **nearly always more efficient** than their counterpart implemented through **Python loops**, especially as the arrays grow in size.

> Any time you see such **a loop in a Python script**, you should consider **whether it can be replaced with a vectorized expression**.



---
## Exploring NumPy's UFuncs

Ufuncs exist in two flavors: **unary ufuncs**, which operate on a single input, and **binary ufuncs**, which operate on two inputs. We'll see examples of both these types of functions here.

### Array arithmetic

NumPy's ufuncs feel very natural to use because they make use of **Python's native arithmetic operators**. The standard addition, subtraction, multiplication, and division can all be used:

In [88]:
x = np.arange(4)
print("x     =", x)
print("x + 5 =", x + 5)
print("x - 5 =", x - 5)
print("x * 2 =", x * 2)
print("x / 2 =", x / 2)
print("x // 2 =", x // 2)  # Floor division

x     = [0 1 2 3]
x + 5 = [5 6 7 8]
x - 5 = [-5 -4 -3 -2]
x * 2 = [0 2 4 6]
x / 2 = [0.  0.5 1.  1.5]
x // 2 = [0 0 1 1]


There is also a unary ufunc for **negation**, a `**` operator for **exponentiation**, and a `%` operator for **modulus**:

In [89]:
print("-x     = ", -x)
print("x ** 2 = ", x ** 2)
print("x % 2  = ", x % 2)

-x     =  [ 0 -1 -2 -3]
x ** 2 =  [0 1 4 9]
x % 2  =  [0 1 0 1]


In addition, these can be **strung together** however you wish, and the **standard order of operations** is respected:

In [90]:
print(x)
-(0.5*x + 1) ** 2

[0 1 2 3]


array([-1.  , -2.25, -4.  , -6.25])

All of these arithmetic operations are simply **convenient wrappers** around specific functions built into NumPy; for example, the `+` operator is a wrapper for the `add` function:

In [91]:
np.add(x, 2)

array([2, 3, 4, 5])

Table 2-2. Arithmetic operators implemented in NumPy

| Operator | Equivalent ufunc | Description |
|----------|------------------|-------------|
| `+` | `np.add` | Addition (e.g., 1 + 1 = 2) |
| `-` | `np.subtract` | Subtraction (e.g., 3 - 2 = 1) |
| `-` | `np.negative` | Unary negation (e.g., -2) |
| `*` | `np.multiply` | Multiplication (e.g., 2 * 3 = 6) |
| `/` | `np.divide` | Division (e.g., 3 / 2 = 1.5) |
| `//` | `np.floor_divide` | Floor division (e.g., 3 // 2 = 1) |
| `**` | `np.power` | Exponentiation (e.g., 2 ** 3 = 8) |
| `%` | `np.mod` | Modulus/remainder (e.g., 9 % 4 = 1) |

Additionally there are Boolean/bitwise operators; we will explore these later.

### Absolute value

Just as NumPy understands Python's built-in arithmetic operators, it also understands Python's built-in **absolute value** function:

In [92]:
x = np.array([-2, -1, 0, 1, 2])
abs(x)

array([2, 1, 0, 1, 2])

The corresponding NumPy ufunc is `np.absolute`, which is also available under the alias `np.abs`:

In [93]:
np.absolute(x)

array([2, 1, 0, 1, 2])

In [94]:
np.abs(x)

array([2, 1, 0, 1, 2])

### Exponents and logarithms

Another common type of operation available in a NumPy ufunc are the **exponentials**:

In [95]:
x = [1, 2, 3]
print("x     =", x)
print("e^x   =", np.exp(x))
print("2^x   =", np.exp2(x))
print("3^x   =", np.power(3, x))

x     = [1, 2, 3]
e^x   = [ 2.71828183  7.3890561  20.08553692]
2^x   = [2. 4. 8.]
3^x   = [ 3  9 27]


The inverse of the exponentials, the **logarithms**, are also available. The basic `np.log` gives the **natural logarithm**; if you prefer to compute the base-2 logarithm or the base-10 logarithm, these are available as well:

In [96]:
x = [1, 2, 4, 10]
print("x        =", x)
print("ln(x)    =", np.log(x))
print("log2(x)  =", np.log2(x))
print("log10(x) =", np.log10(x))

x        = [1, 2, 4, 10]
ln(x)    = [0.         0.69314718 1.38629436 2.30258509]
log2(x)  = [0.         1.         2.         3.32192809]
log10(x) = [0.         0.30103    0.60205999 1.        ]


### Aggregates

For binary ufuncs, there are some interesting **aggregates** that can be computed directly from the object. For example, if we'd like to **reduce** an array with a particular operation, we can use the `reduce` method of any ufunc. A reduce **repeatedly applies a given operation to the elements of an array until only a single result remains**.

For example, calling `reduce` on the `add` ufunc returns the **sum of all elements** in the array:

In [97]:
x = np.arange(1, 6)
print(x)
np.add.reduce(x)

[1 2 3 4 5]


np.int64(15)

Similarly, calling `reduce` on the `multiply` ufunc results in the **product of all array elements**:

In [98]:
np.multiply.reduce(x)

np.int64(120)

If we'd like to **store all the intermediate results** of the computation, we can instead use `accumulate`:

In [99]:
np.add.accumulate(x)

array([ 1,  3,  6, 10, 15])

In [100]:
np.multiply.accumulate(x)

array([  1,   2,   6,  24, 120])

Note that for these particular cases, there are **dedicated NumPy functions** to compute the results (`np.sum`, `np.prod`, `np.cumsum`, `np.cumprod`), which we'll explore.

### Outer products

Finally, any ufunc can compute the **output of all pairs of two different inputs** using the `outer` method. This allows you, in one line, to do things like create a **multiplication table**:

In [101]:
x = np.arange(1, 6)
print(x)
np.multiply.outer(x, x)

[1 2 3 4 5]


array([[ 1,  2,  3,  4,  5],
       [ 2,  4,  6,  8, 10],
       [ 3,  6,  9, 12, 15],
       [ 4,  8, 12, 16, 20],
       [ 5, 10, 15, 20, 25]])


Another extremely useful feature of ufuncs is the ability to operate between arrays of **different sizes and shapes**, a set of operations known as **broadcasting**. This subject is important enough that we will devote a whole section to it.

---
# 5- Aggregations: Min, Max, and Everything in Between

Often when you are faced with a large amount of data, a first step is to compute **summary statistics** for the data in question. Perhaps the most common summary statistics are the **mean** and **standard deviation**, which allow you to summarize the "typical" values in a dataset, but other aggregates are useful as well (the sum, product, median, minimum and maximum, quantiles, etc.).

NumPy has **fast built-in aggregation functions** for working on arrays; we'll discuss and demonstrate some of them here.

## Summing the Values in an Array

As a quick example, consider computing the **sum of all values** in an array. Python itself can do this using the built-in `sum` function:

In [102]:
L = np.random.random(100)
sum(L)

np.float64(50.461758453195614)

The syntax is quite similar to that of NumPy's `sum` function, and the result is the same in the simplest case:

In [103]:
np.sum(L)

np.float64(50.46175845319564)

However, because it executes the operation in **compiled code**, **NumPy's version of the operation** is computed **much more quickly**:

In [104]:
big_array = np.random.rand(1000000)
%timeit sum(big_array)
%timeit np.sum(big_array)

122 ms ± 28.5 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
388 µs ± 13.3 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


Be careful, though: **the `sum` function and the `np.sum` function are not identical**, which can sometimes lead to confusion! In particular, their optional arguments have different meanings, and `np.sum` is aware of **multiple array dimensions**, as we will see in the following section.

## Minimum and Maximum

Similarly, Python has built-in `min` and `max` functions, used to find the **minimum value** and **maximum value** of any given array:

In [105]:
min(big_array), max(big_array)

(np.float64(7.071203171893359e-07), np.float64(0.9999997207656334))

NumPy's corresponding functions have similar syntax, and again operate **much more quickly**:

In [106]:
np.min(big_array), np.max(big_array)

(np.float64(7.071203171893359e-07), np.float64(0.9999997207656334))

In [107]:
%timeit min(big_array)
%timeit np.min(big_array)

75.7 ms ± 15.6 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
458 µs ± 89 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


For `min`, `max`, `sum`, and several other NumPy aggregates, a **shorter syntax** is to use **methods of the array object itself**:

In [108]:
print(big_array.min(), big_array.max(), big_array.sum())

7.071203171893359e-07 0.9999997207656334 500216.8034810001


Whenever possible, make sure that you are using the **NumPy version** of these aggregates when operating on NumPy arrays!

## Multidimensional aggregates

One common type of aggregation operation is an aggregate **along a row or column**. Say you have some data stored in a two-dimensional array:

In [109]:
M = np.random.random((3, 4))
print(M)

[[0.79832448 0.44923861 0.95274259 0.03193135]
 [0.18441813 0.71417358 0.76371195 0.11957117]
 [0.37578601 0.11936151 0.37497044 0.22944653]]


By default, each NumPy aggregation function will return the **aggregate over the entire array**:

In [110]:
M.sum()

np.float64(5.1136763453287335)

Aggregation functions take an additional argument specifying the **axis** along which the aggregate is computed. For example, we can find the **minimum value within each column** by specifying `axis=0`:

In [111]:
M.min(axis=0)  # Min across axis=0, across the rows of each column (min per col)

array([0.18441813, 0.11936151, 0.37497044, 0.03193135])

The function returns **four values**, corresponding to the four columns of numbers.

Similarly, we can find the **maximum value within each row**:

In [112]:
M.max(axis=1)  # Min across axis=1, across the cols of each row (min per row)
# Result is 1D array

array([0.95274259, 0.76371195, 0.37578601])

The way the axis is specified here can be confusing to users coming from other languages. The **axis keyword specifies the dimension of the array that will be collapsed**, *rather than the dimension that will be returned*. So specifying `axis=0` means that the **first axis will be collapsed**: for two-dimensional arrays, this means that values within each column will be aggregated.

## Other aggregation functions

NumPy provides many other aggregation functions, but we won't discuss them in detail here. Additionally, most aggregates have a **NaN-safe counterpart** that computes the result while **ignoring missing values**, which are marked by the special IEEE floating-point `NaN` value. Some of these NaN-safe functions were not added until NumPy 1.8, so they will not be available in older NumPy versions.

**Table 2-3** provides a list of useful aggregation functions available in NumPy.

### Table 2-3. Aggregation functions available in NumPy

| Function Name | NaN-safe Version | Description |
|---------------|------------------|-------------|
| `np.sum` | `np.nansum` | Compute sum of elements |
| `np.prod` | `np.nanprod` | Compute product of elements |
| `np.mean` | `np.nanmean` | Compute mean of elements |
| `np.std` | `np.nanstd` | Compute standard deviation |
| `np.var` | `np.nanvar` | Compute variance |
| `np.min` | `np.nanmin` | Find minimum value |
| `np.max` | `np.nanmax` | Find maximum value |
| `np.argmin` | `np.nanargmin` | Find index of minimum value |
| `np.argmax` | `np.nanargmax` | Find index of maximum value |
| `np.median` | `np.nanmedian` | Compute median of elements |
| `np.percentile` | `np.nanpercentile` | Compute rank-based statistics of elements |
| `np.any` | N/A | Evaluate whether any elements are true |
| `np.all` | N/A | Evaluate whether all elements are true |

We will see these aggregates often throughout the rest of the book.